# Irrigation Water Requirement Prediction Dataset
In this notebook, we train a neural network to predict the amount of irrigation required under varying environmental conditions.

For more information, look up the dataset description at:  
https://www.kaggle.com/datasets/miadul/irrigation-water-requirement-prediction-dataset

-------
## Initialization

In [1]:
# imports
import random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

In [2]:
# load data
df = pd.read_csv("irrigation.csv")

In [3]:
# hyperparameters
RANDOM_STATE = 42
BATCH_SIZE = 32
EPOCHS = 10
LR = 0.01

In [4]:
# label mapping
LABEL_MAP = {'Low': 0, 'Medium': 1, 'High': 2}

-------
## EDA

In [5]:
print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)
print('\nDescribe (numeric):')
print(df.describe())
print('\nDescribe (all):')
print(df.describe(include='all'))

Shape: (10000, 20)

Column dtypes:
Soil_Type                   object
Soil_pH                    float64
Soil_Moisture              float64
Organic_Carbon             float64
Electrical_Conductivity    float64
Temperature_C              float64
Humidity                   float64
Rainfall_mm                float64
Sunlight_Hours             float64
Wind_Speed_kmh             float64
Crop_Type                   object
Crop_Growth_Stage           object
Season                      object
Irrigation_Type             object
Water_Source                object
Field_Area_hectare         float64
Mulching_Used               object
Previous_Irrigation_mm     float64
Region                      object
Irrigation_Need             object
dtype: object

Describe (numeric):
            Soil_pH  Soil_Moisture  Organic_Carbon  Electrical_Conductivity  \
count  10000.000000   10000.000000    10000.000000             10000.000000   
mean       6.487857      36.969207        0.944731                 1.791

In [6]:
# Categorical counts
obj_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()
if obj_cols:
    print('\nCategorical column value counts:')
    for c in obj_cols:
        print(f"\n-- {c} --")
        print(df[c].value_counts(dropna=False))
if bool_cols:
    print('\nBoolean columns:')
    for c in bool_cols:
        print(f"{c}:\n", df[c].value_counts(dropna=False))


Categorical column value counts:

-- Soil_Type --
Soil_Type
Sandy    2536
Silt     2501
Loamy    2486
Clay     2477
Name: count, dtype: int64

-- Crop_Type --
Crop_Type
Rice         1711
Maize        1694
Sugarcane    1678
Potato       1663
Wheat        1659
Cotton       1595
Name: count, dtype: int64

-- Crop_Growth_Stage --
Crop_Growth_Stage
Harvest       2581
Vegetative    2521
Flowering     2465
Sowing        2433
Name: count, dtype: int64

-- Season --
Season
Rabi      3383
Kharif    3362
Zaid      3255
Name: count, dtype: int64

-- Irrigation_Type --
Irrigation_Type
Sprinkler    2527
Rainfed      2511
Drip         2495
Canal        2467
Name: count, dtype: int64

-- Water_Source --
Water_Source
River          2528
Reservoir      2513
Rainwater      2497
Groundwater    2462
Name: count, dtype: int64

-- Mulching_Used --
Mulching_Used
No     5013
Yes    4987
Name: count, dtype: int64

-- Region --
Region
South      2102
West       2017
East       1994
Central    1987
North      19

In [7]:
# Target distribution
if 'Irrigation_Need' in df.columns:
    print('\nTarget distribution (Irrigation_Need):')
    print(df['Irrigation_Need'].value_counts(normalize=False))


Target distribution (Irrigation_Need):
Irrigation_Need
Low       5864
Medium    3800
High       336
Name: count, dtype: int64


-------
## Data wrangling helper functions

In [8]:
# prepare training df with only numeric features
def prepare_numeric(df):
    df = df.copy()
    df = df.dropna()  # drop rows with missing values
    y = df['Irrigation_Need'].map(LABEL_MAP).astype(int)
    X = df.drop(columns=['Irrigation_Need'])  # drop target column
    X_num = X.select_dtypes(include=[np.number])
    return X_num, y

In [9]:
# prepare training df with ALL features (including categoricals)
def prepare_with_categoricals(df):
    df = df.copy()
    df = df.dropna()  # drop rows with missing values
    y = df['Irrigation_Need'].map(LABEL_MAP).astype(int)
    X = df.drop(columns=['Irrigation_Need'])  # drop target column

    # Convert bools to ints
    bool_cols = X.select_dtypes(include=['bool']).columns.tolist()
    for c in bool_cols:
        X[c] = X[c].astype(int)

    # One-hot encode object/category columns
    X = pd.get_dummies(X, drop_first=False)
    return X, y

In [10]:
# for reproducibility
def set_reproducibility(device):
    np.random.seed(RANDOM_STATE)
    random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)
    if device.type == 'cuda':
        torch.cuda.manual_seed(RANDOM_STATE)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

-------
## Training the neural network

In [11]:
## model definition
class SmallNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 10),
            nn.ELU(),
            nn.Linear(10, 15),
            nn.ELU(),
            nn.Linear(15, 3)  # 3 classes
        )

    def forward(self, x):
        return self.net(x)

In [12]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def load_best_model(self, model):
        model.load_state_dict(self.best_model_state)

In [13]:
def train_val_test_split(X, y):
    # Split: 60% train, 20% val, 20% test
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=0.25, random_state=RANDOM_STATE, stratify=y_trainval)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    # Convert to tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.long)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.long)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test.values, dtype=torch.long)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)
    test_ds = TensorDataset(X_test_t, y_test_t)
    
    return train_ds, val_ds, test_ds

In [14]:
def train_and_validate(X, y, model, device, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, weight_decay=0.0, momentum=0.9, balance_class_weights=False, early_stopping_patience=EPOCHS):
    train_ds, val_ds, test_ds = train_val_test_split(X, y)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=(device.type=='cuda'))
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=(device.type=='cuda'))

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(momentum, 0.999))
    if balance_class_weights:
        class_weights = []
        for i in range(3):
            prop_i = sum([int(x[1])==i for x in train_ds]) / len(train_ds)
            w_i = 1 - prop_i  # weight is higher for less frequent classes
            w_i = round(w_i, 3)
            class_weights.append(w_i)
        print(f"Class weights: {class_weights}")
        class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    else:
        criterion = nn.CrossEntropyLoss()

    if early_stopping_patience < EPOCHS:
        early_stopping = EarlyStopping(patience=early_stopping_patience)

    print(f"\nTraining model (input_dim={X.shape[1]}) on device: {device}")

    for epoch in range(1, epochs+1):
        model.train()
        running_loss = 0.0
        for xb, yb in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        avg_train_loss = running_loss / len(train_loader.dataset)

        # Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
                preds = logits.argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += xb.size(0)
        avg_val_loss = val_loss / len(val_loader.dataset)
        val_acc = correct / total if total>0 else 0.0
        print(f"Epoch {epoch}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, Val Acc={val_acc:.4f}")

        if early_stopping_patience < EPOCHS:
            early_stopping(avg_val_loss, model)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch}.")
                early_stopping.load_best_model(model)
                break

    # Return model and test dataset for final evaluation
    return model, test_ds

In [15]:
# Let's determine the device to use (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [16]:
# 1) Numeric-only features
print('\n\n--- Training with numeric-only features ---')
set_reproducibility(device)
X_num, y_num = prepare_numeric(df)
print('Numeric feature columns:', X_num.columns.tolist())
if X_num.shape[1] == 0:
    print('No numeric features found. Skipping numeric-only training.')
else:
    net = SmallNN(input_dim=X_num.shape[1]).to(device)
    model_num, test_ds_num = train_and_validate(X_num, y_num, net, device)



--- Training with numeric-only features ---
Numeric feature columns: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']

Training model (input_dim=11) on device: cuda


Epoch 1/10: 100%|██████████| 188/188 [00:00<00:00, 340.83it/s]


Epoch 1: Train Loss=0.6779, Val Loss=0.6390, Val Acc=0.6740


Epoch 2/10: 100%|██████████| 188/188 [00:00<00:00, 463.52it/s]


Epoch 2: Train Loss=0.6397, Val Loss=0.6191, Val Acc=0.6965


...

Epoch 8: Train Loss=0.5785, Val Loss=0.5920, Val Acc=0.7230


Epoch 9/10: 100%|██████████| 188/188 [00:00<00:00, 483.48it/s]


Epoch 9: Train Loss=0.5755, Val Loss=0.5894, Val Acc=0.7180


Epoch 10/10: 100%|██████████| 188/188 [00:00<00:00, 470.04it/s]

Epoch 10: Train Loss=0.5730, Val Loss=0.5813, Val Acc=0.7300


In [17]:
# 2) Include categorical features (one-hot)
print('\n\n--- Training including categorical features (one-hot encoded) ---')
set_reproducibility(device)
X_all, y_all = prepare_with_categoricals(df)
print('Feature columns after encoding:', X_all.shape[1])
net = SmallNN(input_dim=X_all.shape[1]).to(device)
model_all, test_ds_all = train_and_validate(X_all, y_all, net, device)



--- Training including categorical features (one-hot encoded) ---
Feature columns after encoding: 43

Training model (input_dim=43) on device: cuda


Epoch 1/10: 100%|██████████| 188/188 [00:00<00:00, 430.11it/s]


Epoch 1: Train Loss=0.4669, Val Loss=0.4097, Val Acc=0.8200


Epoch 2/10: 100%|██████████| 188/188 [00:00<00:00, 461.01it/s]


Epoch 2: Train Loss=0.3972, Val Loss=0.4041, Val Acc=0.8150


...

Epoch 8: Train Loss=0.3153, Val Loss=0.3604, Val Acc=0.8370


Epoch 9/10: 100%|██████████| 188/188 [00:00<00:00, 440.42it/s]


Epoch 9: Train Loss=0.3086, Val Loss=0.3442, Val Acc=0.8460


Epoch 10/10: 100%|██████████| 188/188 [00:00<00:00, 433.59it/s]


Epoch 10: Train Loss=0.3010, Val Loss=0.3241, Val Acc=0.8500


Additionally including the categorical features into the model significantly improved the model performance as per the loss and accuracy on the validation. As such, we'll keep this model for evaluation.

-------
## Evaluation

In [18]:
def evaluate_model(model, test_ds, device):
    model.eval()
    X_test_t, y_test_t = test_ds.tensors
    X_test_t = X_test_t.to(device)
    y_test_t = y_test_t.to(device)
    with torch.no_grad():
        logits = model(X_test_t)
        preds = logits.argmax(dim=1).cpu().numpy()
        truths = y_test_t.cpu().numpy()

    y_true = truths
    y_pred = preds
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("\nClassification Report:")
    classes = ['Low', 'Medium', 'High']
    print(classification_report(y_true, y_pred, target_names=classes))
    print('\nConfusion Matrix (rows=true, cols=pred):')
    print(confusion_matrix(y_true, y_pred))

In [19]:
evaluate_model(model_num, test_ds_num, device)

Accuracy: 0.7135

Classification Report:
              precision    recall  f1-score   support

         Low       0.77      0.81      0.79      1173
      Medium       0.63      0.61      0.62       760
        High       0.67      0.18      0.28        67

    accuracy                           0.71      2000
   macro avg       0.69      0.53      0.56      2000
weighted avg       0.71      0.71      0.71      2000


Confusion Matrix (rows=true, cols=pred):
[[952 221   0]
 [291 463   6]
 [  0  55  12]]


In [20]:
evaluate_model(model_all, test_ds_all, device)

Accuracy: 0.849

Classification Report:
              precision    recall  f1-score   support

         Low       0.89      0.90      0.89      1173
      Medium       0.81      0.78      0.80       760
        High       0.58      0.73      0.65        67

    accuracy                           0.85      2000
   macro avg       0.76      0.80      0.78      2000
weighted avg       0.85      0.85      0.85      2000


Confusion Matrix (rows=true, cols=pred):
[[1055  117    1]
 [ 132  594   34]
 [   0   18   49]]


-------
# Encore (Class imbalance)

In [21]:
print('\n\n--- Training with Balanced Class Weights ---')
set_reproducibility(device)
# X_all, y_all = prepare_with_categoricals(df)
print('Feature columns after encoding:', X_all.shape[1])
net = SmallNN(input_dim=X_all.shape[1]).to(device)
model_bal, test_ds_bal = train_and_validate(X_all, y_all, net, device, balance_class_weights=True)



--- Training with Balanced Class Weights ---
Feature columns after encoding: 43
Class weights: [0.414, 0.62, 0.966]

Training model (input_dim=43) on device: cuda


Epoch 1/10: 100%|██████████| 188/188 [00:00<00:00, 416.35it/s]


Epoch 1: Train Loss=0.5168, Val Loss=0.4440, Val Acc=0.8245


Epoch 2/10: 100%|██████████| 188/188 [00:00<00:00, 431.13it/s]


Epoch 2: Train Loss=0.4389, Val Loss=0.4480, Val Acc=0.8075


...

Epoch 8: Train Loss=0.3406, Val Loss=0.3956, Val Acc=0.8370


Epoch 9/10: 100%|██████████| 188/188 [00:00<00:00, 434.42it/s]


Epoch 9: Train Loss=0.3365, Val Loss=0.3637, Val Acc=0.8420


Epoch 10/10: 100%|██████████| 188/188 [00:00<00:00, 445.53it/s]


Epoch 10: Train Loss=0.3249, Val Loss=0.3658, Val Acc=0.8535


In [22]:
evaluate_model(model_bal, test_ds_bal, device)

Accuracy: 0.85

Classification Report:
              precision    recall  f1-score   support

         Low       0.90      0.89      0.89      1173
      Medium       0.81      0.80      0.80       760
        High       0.59      0.81      0.68        67

    accuracy                           0.85      2000
   macro avg       0.77      0.83      0.79      2000
weighted avg       0.85      0.85      0.85      2000


Confusion Matrix (rows=true, cols=pred):
[[1041  132    0]
 [ 118  605   37]
 [   0   13   54]]


-------
# Encore (Kaiming)

In [23]:
import torch.nn.init as init

## model definition
## same as before but with kaiming initialization
class SmallNN_kaiming(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 10),
            nn.ELU(),
            nn.Linear(10, 15),
            nn.ELU(),
            nn.Linear(15, 3)  # 3 classes
        )
        # Initialize weights with Kaiming initialization
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                # init.kaiming_normal_(layer.weight)
                init.kaiming_uniform_(layer.weight)

    def forward(self, x):
        return self.net(x)

In [24]:
print('\n\n--- Training with Kaiming Initialization ---')
set_reproducibility(device)
# X_all, y_all = prepare_with_categoricals(df)
print('Feature columns after encoding:', X_all.shape[1])
net = SmallNN_kaiming(input_dim=X_all.shape[1]).to(device)
model_kmng, test_ds_kmng = train_and_validate(X_all, y_all, net, device, balance_class_weights=True)



--- Training with Kaiming Initialization ---
Feature columns after encoding: 43
Class weights: [0.414, 0.62, 0.966]

Training model (input_dim=43) on device: cuda


Epoch 1/10: 100%|██████████| 188/188 [00:00<00:00, 425.03it/s]


Epoch 1: Train Loss=0.6208, Val Loss=0.4640, Val Acc=0.8045


Epoch 2/10: 100%|██████████| 188/188 [00:00<00:00, 431.21it/s]


Epoch 2: Train Loss=0.4420, Val Loss=0.4526, Val Acc=0.8215


...

Epoch 8: Train Loss=0.2517, Val Loss=0.3004, Val Acc=0.8750


Epoch 9/10: 100%|██████████| 188/188 [00:00<00:00, 414.24it/s]


Epoch 9: Train Loss=0.2417, Val Loss=0.2641, Val Acc=0.9005


Epoch 10/10: 100%|██████████| 188/188 [00:00<00:00, 420.13it/s]


Epoch 10: Train Loss=0.2196, Val Loss=0.3026, Val Acc=0.8935


In [25]:
evaluate_model(model_kmng, test_ds_kmng, device)

Accuracy: 0.889

Classification Report:
              precision    recall  f1-score   support

         Low       0.91      0.94      0.92      1173
      Medium       0.87      0.84      0.85       760
        High       0.75      0.67      0.71        67

    accuracy                           0.89      2000
   macro avg       0.84      0.81      0.83      2000
weighted avg       0.89      0.89      0.89      2000


Confusion Matrix (rows=true, cols=pred):
[[1097   76    0]
 [ 109  636   15]
 [   0   22   45]]


-------
# Encore (Early stopping)

In [26]:
print('\n\n--- Training with Early Stopping ---')
set_reproducibility(device)
# X_all, y_all = prepare_with_categoricals(df)
print('Feature columns after encoding:', X_all.shape[1])
net = SmallNN_kaiming(input_dim=X_all.shape[1]).to(device)
model_es, test_ds_es = train_and_validate(X_all, y_all, net, device, balance_class_weights=True, lr=0.001, early_stopping_patience=3, epochs=100)



--- Training with Early Stopping ---
Feature columns after encoding: 43
Class weights: [0.414, 0.62, 0.966]

Training model (input_dim=43) on device: cuda


Epoch 1/100: 100%|██████████| 188/188 [00:00<00:00, 429.22it/s]


Epoch 1: Train Loss=1.3135, Val Loss=0.8014, Val Acc=0.6735


Epoch 2/100: 100%|██████████| 188/188 [00:00<00:00, 417.30it/s]


Epoch 2: Train Loss=0.6721, Val Loss=0.5986, Val Acc=0.7855


...

Epoch 72: Train Loss=0.1800, Val Loss=0.2550, Val Acc=0.9040


Epoch 73/100: 100%|██████████| 188/188 [00:00<00:00, 416.43it/s]


Epoch 73: Train Loss=0.1785, Val Loss=0.2544, Val Acc=0.9080


Epoch 74/100: 100%|██████████| 188/188 [00:00<00:00, 405.50it/s]


Epoch 74: Train Loss=0.1774, Val Loss=0.2496, Val Acc=0.9045
Early stopping triggered at epoch 74.


In [27]:
evaluate_model(model_es, test_ds_es, device)

Accuracy: 0.896

Classification Report:
              precision    recall  f1-score   support

         Low       0.95      0.91      0.93      1173
      Medium       0.84      0.90      0.87       760
        High       0.70      0.67      0.69        67

    accuracy                           0.90      2000
   macro avg       0.83      0.83      0.83      2000
weighted avg       0.90      0.90      0.90      2000


Confusion Matrix (rows=true, cols=pred):
[[1064  109    0]
 [  58  683   19]
 [   0   22   45]]
